# VQCC Fast Runner — locally verified 27.3151128090

This portfolio uses public quantum circuits with a further chemistry weight refit. The original official local verifiers returned PASS on **200,540,928 InvSqrt shots**, **102,400 DQI shots** (zero value and phase errors), and **78,080 chemistry shots**. The runner re-verifies chemistry and checks SHA-256 for all 18 final members before writing `submission.zip`.

| Task | Cost | Q |
|---|---:|---:|
| InvSqrt | 78,673.56921101942 | 46 |
| DQI, zero errors | 360,209,905,486.0156 | 889 |
| Chemistry | 294,888,331.7976092 | 427 |

The geometric mean of reference/candidate ratios is **27.315112809026374** with the verified zero-error DQI cost. A public score is determined by the competition draw and may differ. The chemistry improvement over the public starting factor is about 0.30665 cost units, so it is unlikely to change the five-decimal score.

> ### 🔬 Cycle #105 Research Focus: Cholesky Vector Truncation & Low-Rank Auxiliary Compression
> **Category:** `LOW-RANK FACTORIZATION` • **Updated:** `2026-11-15 09:52:42 UTC`
> 
> Cholesky decomposition of two-electron integrals yields auxiliary vectors $L_{ik}^{(\gamma)}$. Eigenvalue thresholding selects the minimal basis rank needed to preserve chemical fidelity while slashing state preparation Toffoli counts.

## Method and verification

The public portfolio is from [Arizal Muluk](https://www.kaggle.com/code/arizalfirdaus123/vqcc-analytic-gradient-df-thc-mbuc-optimization), using manwithacat's InvSqrt, Areeb Ali's DQI and Maoxin123's coded-word chemistry. Our local improvement refits only `W` and `H2_I` with [prvsiyan's Apache-2.0 reduced-gradient optimizer](https://www.kaggle.com/code/prvsiyan/verifier-guided-quantum-circuit-optimization). The angle words used for optimization were recovered from the actual loader circuits by the original verifier: the optional angle-word array in the source NPZ differs in 473 entries. The circuits and original NPZ angle array are preserved in the submission.

All final circuit files are byte-identical to the public portfolio; only the factor NPZ changes. A compressed XOR patch reconstructs that exact verified NPZ and is checked by hash. Its physics is close to the official error and energy thresholds, so this notebook reruns the original chemistry verifier before packaging. InvSqrt and DQI results are prior independent local verifications of exactly matching circuit hashes.

Credits: [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook), [manwithacat](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry), [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-30692), [Maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4030-toffoli-coded-word-chemistry), [Arizal Muluk](https://www.kaggle.com/code/arizalfirdaus123/vqcc-analytic-gradient-df-thc-mbuc-optimization), prvsiyan, Ertuğrul Özer and upstream authors. Component licences remain in effect.


In [1]:
import pathlib,hashlib,zipfile,lzma,base64,json,os,sys,subprocess,shutil
INPUT=pathlib.Path(os.environ.get('VQCC_INPUT','/kaggle/input'));WORK=pathlib.Path(os.environ.get('VQCC_WORK','/kaggle/working'));WORK.mkdir(parents=True,exist_ok=True)
PINS={'invsqrt.kmb': '1ba197b027713aac164056d12505c87f286c3fcd94f403c522073325763bf2f9', 'dqi.kmb': '821bf8d157c03c3e868a0099b0ab1438cfba664ab1bc6fec3210654099b672a3', 'dfthcsos/angle_k_0.kmb': '71735b23037e8a64d16e39b3347d440b36bdf8813613ec70c36a85bfd362d7c2', 'dfthcsos/angle_k_1.kmb': '1a43bbc368785aa5fc5210f6e4d5d2f9ddf114f7f3f6d3e3964f9478056bb4be', 'dfthcsos/angle_k_10.kmb': '5fea9f297afea7ffe74054ed1b44f92031a5f6e319a7f1c8741f86c05b234834', 'dfthcsos/angle_k_11.kmb': 'dc425931348cc4950931c400ab19322e01495b7cf3e2d02bafabcaaf9559d28a', 'dfthcsos/angle_k_12.kmb': '64bbada35a382068b8dfbca0df0d781df8704aa367882cc426b76e2767f3d13f', 'dfthcsos/angle_k_13.kmb': '958789349187a70ab3baf426655aacd2360113f71ab9a7b49d6624bd0e20bcd6', 'dfthcsos/angle_k_14.kmb': 'f35c803dece96ceb0ba24040e7c232a73170a3692c4f5212bdeadcbfcd154212', 'dfthcsos/angle_k_2.kmb': '0970b392fb4bed09d6026b77b4b31510573123cb76955030115507bffde4a6bb', 'dfthcsos/angle_k_3.kmb': 'f9d313326dd07028bf6ef672bb47013f01e9311695d66b8d1ee2546d7dbacd7f', 'dfthcsos/angle_k_4.kmb': '0ab29f06e44ef247035c971dcca30e27616bdfdc2a102fbb58c3882bbfbfd1e0', 'dfthcsos/angle_k_5.kmb': 'b346df2edafbaf4d95192a974db27abf71b770f51ee61dae2eb352f422cefdf4', 'dfthcsos/angle_k_6.kmb': '749d9edb817c59f8a3507400002e75c2d8db55932a6933db5c5f6f59a2679f7b', 'dfthcsos/angle_k_7.kmb': '0ad7d3775a36d7c14e3d4dfa4a144267e4e9759521fae4022715be25c309638e', 'dfthcsos/angle_k_8.kmb': '46e0f8d69e532438d476aa8244f257a649fdc4cffe9d41f9921758f4dc18fa83', 'dfthcsos/angle_k_9.kmb': '0b995537e919351a705babb6294dfe3525ef52a3c40b08122d842f45df8182da', 'dfthcsos/submission.npz': '4df35abdeec320a3f1c10f6f533e73550e50bb45d95e666deaa01062810a3559'}
def sha(b):return hashlib.sha256(b).hexdigest()
source=None
for p in INPUT.rglob('submission.zip'):
 try:
  with zipfile.ZipFile(p) as z:
   if set(z.namelist())==set(PINS) and sha(z.read('invsqrt.kmb'))==PINS['invsqrt.kmb'] and sha(z.read('dfthcsos/submission.npz'))==PINS['dfthcsos/submission.npz']:source=p;break
 except (OSError,zipfile.BadZipFile):continue
assert source,'Pinned public source archive missing'
root=WORK/'portfolio_273151128';root.mkdir(exist_ok=True)
with zipfile.ZipFile(source) as z:
 for n,h in PINS.items():
  data=z.read(n);assert sha(data)==h,n
  p=root/n;p.parent.mkdir(exist_ok=True,parents=True);p.write_bytes(data)
p=root/'dfthcsos/submission.npz';old=p.read_bytes();delta=lzma.decompress(base64.b64decode('...'));assert len(old)==len(delta)
p.write_bytes(bytes(a^b for a,b in zip(old,delta)));PINS['dfthcsos/submission.npz']='3b7d0cd9068d1d00d31eef128e8c66233ec3a1e61540829e099b9800e266d8cb';assert sha(p.read_bytes())==PINS['dfthcsos/submission.npz']
for n,h in PINS.items():assert sha((root/n).read_bytes())==h,n
comp=next(p.parents[2] for p in INPUT.rglob('FeMoCo76o113e.fcidump') if p.parent.name=='dfthcsos' and p.parent.parent.name=='verifiers')
env=dict(os.environ,PYTHONPATH=f'{comp/"kickmix"}:{comp/"verifiers"}',OPENBLAS_NUM_THREADS='1')
r=subprocess.run([sys.executable,str(comp/'verifiers/dfthcsos/verify.py'),str(root/'dfthcsos')],capture_output=True,text=True,env=env,timeout=3600);assert r.returncode==0,(r.stdout[-3000:],r.stderr[-3000:])
chem=json.loads(r.stdout[r.stdout.index('{'):]);assert chem['status']=='PASS'
inv=78673.56921101942;dqi=360209905486.0156
gm=((2076774.233715282/inv)*(24439463583672.93/dqi)*(3355599588.3155365/chem['score']))**(1/3)
report={'chemistry':chem,'invsqrt_cost_from_local_official_verifier':inv,'dqi_zero_error_cost_from_local_official_verifier':dqi,'conditional_zero_error_gm':gm,'pins':PINS,'source':str(source),'scope':'Chemistry freshly verified; InvSqrt and DQI costs from earlier local official verification of identical hashes.'}
(WORK/'verification.json').write_text(json.dumps(report,indent=2));print('Chemistry',chem['status'],chem['score'],'conditional GM',gm)
with zipfile.ZipFile(WORK/'submission.zip','w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
 for n in sorted(PINS):z.write(root/n,n)
print('Pinned 18-member submission.zip ready')


Chemistry PASS 294888331.7976092 conditional GM 27.315112809026374
Pinned 18-member submission.zip ready
